<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook E01: An End-to-End Forecasting Pipeline</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook E01: An End-to-End Forecasting Pipeline](../notebooks/E01_End_to_end_pipeline.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The Belgian load series, the feature set and the daily evaluation from the notebook. Both exercises use
the feature-based model only, so this notebook runs on the default install — no `dl` group needed.

In [ ]:
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "BE") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .asfreq("h")
)

TRAIN_END = "2019-12-31"
HORIZON = 24

BELGIAN_HOLIDAYS = pd.to_datetime([
    "2015-01-01", "2015-04-06", "2015-05-01", "2015-05-14", "2015-05-25", "2015-07-21",
    "2015-08-15", "2015-11-01", "2015-11-11", "2015-12-25",
    "2016-01-01", "2016-03-28", "2016-05-01", "2016-05-05", "2016-05-16", "2016-07-21",
    "2016-08-15", "2016-11-01", "2016-11-11", "2016-12-25",
    "2017-01-01", "2017-04-17", "2017-05-01", "2017-05-25", "2017-06-05", "2017-07-21",
    "2017-08-15", "2017-11-01", "2017-11-11", "2017-12-25",
    "2018-01-01", "2018-04-02", "2018-05-01", "2018-05-10", "2018-05-21", "2018-07-21",
    "2018-08-15", "2018-11-01", "2018-11-11", "2018-12-25",
    "2019-01-01", "2019-04-22", "2019-05-01", "2019-05-30", "2019-06-10", "2019-07-21",
    "2019-08-15", "2019-11-01", "2019-11-11", "2019-12-25",
    "2020-01-01", "2020-04-13", "2020-05-01", "2020-05-21", "2020-06-01", "2020-07-21",
    "2020-08-15", "2020-11-01", "2020-11-11", "2020-12-25",
])


def build_features(series):
    """Everything knowable the day before, and nothing else."""
    features = pd.DataFrame(index=series.index)

    for lag in (24, 25, 48, 168, 169):
        features[f"lag_{lag}"] = series.shift(lag)

    features["roll_mean_168"] = series.shift(24).rolling(168).mean()

    features["hour"] = series.index.hour
    features["day_of_week"] = series.index.dayofweek
    features["month"] = series.index.month
    features["day_of_year"] = series.index.dayofyear
    features["is_weekend"] = (series.index.dayofweek >= 5).astype(int)
    features["is_holiday"] = series.index.normalize().isin(BELGIAN_HOLIDAYS).astype(int)

    return features


def new_model():
    return lgb.LGBMRegressor(
        n_estimators=600, learning_rate=0.05, num_leaves=63, random_state=0, verbose=-1
    )


features = build_features(load)
usable = features.notna().all(axis=1) & load.notna()

model = new_model().fit(
    features[usable & (features.index <= TRAIN_END)],
    load[usable & (load.index <= TRAIN_END)],
)

LAST_FULL_DAY = load.index.max().normalize() - pd.Timedelta(days=1)
ORIGINS = pd.date_range("2020-01-01", LAST_FULL_DAY, freq="D")


def evaluate_daily(forecast_function):
    """One forecast per day of 2020, keeping both the size and the sign of the error."""
    records = []

    for origin in ORIGINS:
        actual = load.loc[origin:origin + pd.Timedelta(hours=HORIZON - 1)]
        if len(actual) < HORIZON:
            continue

        predicted = forecast_function(origin)
        if predicted is None:
            continue
        predicted = np.asarray(predicted)[:HORIZON]

        records.append({
            "date": origin,
            "mae": mean_absolute_error(actual.values, predicted),
            "bias": float(np.mean(predicted - actual.values)),
        })

    return pd.DataFrame(records).set_index("date")


def static_forecast(origin):
    """The notebook's model: fitted once on data up to the end of 2019."""
    wanted = pd.date_range(origin, periods=HORIZON, freq="h")
    window = features.reindex(wanted)
    return None if window.isna().any().any() else model.predict(window)


static = evaluate_daily(static_forecast)
print(f"LightGBM over {len(static)} days of 2020: MAE {static['mae'].mean():.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Is LightGBM's collapse a *level* problem? For each month of 2020, compute the mean signed error (forecast minus actual) alongside the MAE. Then try correcting it: before each day's forecast, subtract the model's mean error over the previous seven days, and re-score. How much of April does that recover, and what does it do in a month with no break?

In [ ]:
monthly = static.resample("MS").mean()
monthly["bias share of MAE"] = monthly["bias"].abs() / monthly["mae"]

print("Signed error (forecast - actual) against absolute error\n")
print(monthly.round(2).to_string())

**It is almost entirely a level problem.** In April the MAE is 461.9 MW and the mean signed error is
**+425.3** — 92% of the error is the model forecasting too high, uniformly.

That number is worth pausing on. An unbiased forecaster's errors cancel: it is too high on some hours and
too low on others, so the mean signed error is near zero while the MAE is not. April's ratio of 0.92 says
the opposite. The model is not confused about the *shape* of a day — when demand rises, when it peaks,
how the weekend differs. It has the shape right and the **level** wrong, by about 425 MW on a mean demand of
roughly 9,800.

That is exactly what a structural break looks like from inside a model. Belgian demand dropped by about a
tenth and stayed there; every feature the model has — lags, rolling mean, calendar — describes a world
where it had not. Look down the `bias share of MAE` column and you can see the lockdown arrive and fade:
0.5 in January, 0.2 in February, then 0.7, **0.9**, 0.8 through March, April and May, easing back to 0.3
by July.

A diagnosis like that suggests its own fix. If the error is a constant offset, measure the offset and
subtract it.

In [ ]:
def bias_corrected_forecast(history, window=7):
    """Subtract the mean error of the previous `window` days from each forecast.

    `history` is the record of what the uncorrected model already did, which in
    deployment is simply the log of yesterday's forecasts against what happened.
    Only days strictly before the origin are used, so nothing looks forward.
    """
    def forecast(origin):
        raw = static_forecast(origin)
        if raw is None:
            return None

        recent = history.loc[history.index < origin, "bias"].tail(window)
        offset = recent.mean() if len(recent) == window else 0.0
        return raw - offset

    return forecast


corrected = evaluate_daily(bias_corrected_forecast(static))

comparison = pd.DataFrame({
    "uncorrected": static["mae"].resample("MS").mean(),
    "corrected": corrected["mae"].resample("MS").mean(),
})
comparison["recovered"] = comparison["uncorrected"] - comparison["corrected"]

print(comparison.round(1).to_string())
print()
print(f"Whole of 2020   uncorrected {static['mae'].mean():.1f} MW"
      f"   corrected {corrected['mae'].mean():.1f} MW")
print(f"N-BEATS, for reference: 248.2 MW")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].plot(comparison.index, comparison["uncorrected"], marker="o", markersize=5,
             linewidth=1.6, color="crimson", label="LightGBM")
axes[0].plot(comparison.index, comparison["corrected"], marker="o", markersize=5,
             linewidth=1.6, color="seagreen", label="LightGBM, bias corrected")
axes[0].axhline(248.2, color="steelblue", linestyle="--", linewidth=1.2, label="N-BEATS (whole year)")
axes[0].axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-05-31"),
                color="crimson", alpha=0.08)
axes[0].set_title("Monthly MAE", fontsize=13, fontweight="bold")
axes[0].set_ylabel("MAE (MW)")
axes[0].legend(fontsize=9)
axes[0].tick_params(axis="x", rotation=30)

axes[1].plot(static.index, static["bias"].rolling(7).mean(), color="crimson", linewidth=1.3,
             label="uncorrected")
axes[1].plot(corrected.index, corrected["bias"].rolling(7).mean(), color="seagreen",
             linewidth=1.3, label="corrected")
axes[1].axhline(0, color="black", linewidth=1.0)
axes[1].axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-05-31"),
                color="crimson", alpha=0.08)
axes[1].set_title("Bias, 7-day rolling mean", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Forecast - actual (MW)")
axes[1].legend(fontsize=9)
axes[1].tick_params(axis="x", rotation=30)

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**It recovers 189 of April's 462 MW**, taking the worst month from 461.9 to 272.8 — which is better
than N-BEATS managed in April (275). Over the whole of 2020 it takes LightGBM from 285.5 to **248.3 MW**,
against N-BEATS's 248.2.

**Three lines of arithmetic, applied to a model the notebook had written off, close the entire gap.**

And to the second half of the question: **in a quiet month it costs you.** February, with no break to
correct, gets *worse* — 277.2 becomes 305.4, a loss of 28 MW. The correction is estimated from seven days
of residuals, and when there is no real bias to find, what it finds is noise, which it then subtracts from
a forecast that did not need it.

That trade is the whole content of the exercise:

- **When the level has genuinely moved**, a correction estimated from recent residuals is the fastest
  instrument available. It responds within about a week, which is roughly the length of its window.
- **When nothing has moved**, it adds a small amount of variance for no gain.

Over 2020 the trade is strongly positive because the break was large and lasted months. In a year without
a break it would have been mildly negative. If you deploy one, gate it: apply the correction only when the
recent bias is large relative to its own historical spread, which is the "trigger" that section 8 of the
notebook asks for.

**What this does to the notebook's conclusion** is the part worth arguing about. Section 8 recommends
N-BEATS partly because LightGBM "demonstrated how it fails". It did — but the failure turns out to be the
most benign kind there is: **visible in the residuals, diagnosable in one table, and correctable without
retraining anything.** A model that fails by drifting off-level and tells you so in its own error log is
in a much better position than one that fails unpredictably.

The honest revision is not "LightGBM wins" — the corrected score ties, and ties are not wins. It is that
the gap the notebook used to justify the more complex model was not really about the models at all. It was
about one of them being given a mechanism to adapt and the other not.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> The other obvious response to a structural break is retraining. Refit the model at the start of each month on everything available up to that point, and score the same way. How much of April does retraining recover compared with the correction above, and why is the difference so large?

In [ ]:
refitted = {}


def monthly_refit_forecast(origin):
    """Refit on everything up to the day before the month starts."""
    month = (origin.year, origin.month)

    if month not in refitted:
        cutoff = origin.normalize() - pd.Timedelta(days=1)
        available = usable & (features.index <= cutoff)
        refitted[month] = new_model().fit(features[available], load[available])

    wanted = pd.date_range(origin, periods=HORIZON, freq="h")
    window = features.reindex(wanted)
    return None if window.isna().any().any() else refitted[month].predict(window)


started = time.time()
refit = evaluate_daily(monthly_refit_forecast)

strategies = pd.DataFrame({
    "no adaptation": static["mae"].resample("MS").mean(),
    "refit monthly": refit["mae"].resample("MS").mean(),
    "bias corrected": corrected["mae"].resample("MS").mean(),
})

print(f"{len(refitted)} refits in {time.time() - started:.0f}s\n")
print(strategies.round(1).to_string())
print()
print(strategies.mean().round(1).to_string())

**Retraining recovers 51 MW of April's 462; the bias correction recovered 189.** Monthly refitting takes
the year from 285.5 to 274.9, against the correction's 248.3. It is roughly **a quarter as effective**,
and it costs nine model fits instead of an arithmetic mean.

The reason is worth stating carefully, because "retrain when things change" is such standard advice that
its failure here looks like a bug.

**Ask what the April model was actually trained on.** It sees every hour from 2015 to 31 March 2020 —
about 45,000 rows. The break began in the middle of March. So the new regime is represented by roughly two
weeks, **under 1% of the training data**, and the other 99% describes the world as it was. A gradient
boosting model fits the bulk of its data. Two weeks of anomaly among five years is indistinguishable from
the sort of unusual fortnight that any five-year history contains, and the model correctly declines to
rebuild itself around it.

And exactly one month later it catches up. In April the refit is far behind the correction, 411.2 against
272.8; in May, with six weeks of post-break data instead of two, the two are level — 234.3 against 235.2.
Retraining works. It is simply **one month slower than the break**, and April is the month that cost.

Set the two mechanisms side by side and the difference is one of timescale:

| | what it uses | how fast it responds | April recovery |
|---|---|---|---|
| **Monthly refit** | all history, reweighted by one extra month | months | 51 MW |
| **Bias correction** | the last 7 days of residuals | about a week | 189 MW |

**Neither is a substitute for the other, and they are not really competing.** Retraining is how a model
absorbs a change permanently, so that the new level becomes what it expects rather than something being
subtracted from it. A residual correction is how it survives the interval between the change happening and
the retraining catching up. In a deployed system you want both, which is the sharpened version of the
advice in section 8: **retrain on a schedule, correct on a trigger, and do not expect the schedule to do
the trigger's job.**

Two smaller observations from the table, both of which generalise:

- **Refitting is not free even when nothing is wrong.** June, July and August all get slightly worse
  (-13, -3, -6 MW). Each refit is a new model with its own idiosyncrasies, and in a stable period the
  incumbent is already as good as it is going to get. Retraining on a schedule buys insurance, and
  insurance has a premium.
- **September is the largest quiet-period gain**, 241.4 to 216.5, because by then the extra training data
  includes the whole of the post-break world. That is retraining finally doing its job, six months after
  the event it was responding to.

---

Back to [Notebook E01](../notebooks/E01_End_to_end_pipeline.ipynb). That is the end of the course
material; the appendices in Part F cover the datasets.